In [1]:
!pip install transformers
!pip install "transformers[torch]"

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/11.7 MB 12.0 MB/s eta 0:00:01
   ----------- ---------------------------- 3.4/11.7 MB 9.6 MB/s eta 0:00:01
   ----------------- ---------------------- 5.2/11.7 MB 9.7 MB/s eta 0:00:01
   ------------------ --------------------- 5.5/11.7 MB 7.6 MB/s eta 0:00:01
   -------------------- ------------------- 6.0/11.7 MB 6.5 MB/s eta 0:00:01
   ---------------------- ----------------- 6.6/11.7 MB 5.9 MB/s eta 0:00:01
   ------------------------ --------------- 7.3/11.7 MB 5.1 MB/s eta 0:00:01
   ------------------------- -------------- 7.6/11.7 MB 4.9 MB/s eta 0:00:01
   --------------------------- ------------ 8.1/11.7 MB 4.4 MB/s eta 0:00:01
   ----------------------------- ---------- 8.7/11.7 MB 4.2 MB/s eta 0:00:01
   ------------------------------ --------- 8.9/11.7 MB 4.0 MB/s eta 0:00:01
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments,T5ForConditionalGeneration

C:\Users\kkris\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.shape

(14732, 3)

In [4]:
val_data.shape

(818, 3)

In [5]:
#Random Sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)
train_data.shape

(4000, 3)

In [6]:
#Data PreProcessing

import re
def clean_data(text):
    text = re.sub(r"\r\n"," ",text)
    text = re.sub(r"\s+"," ",text)
    text = re.sub(r"<.*?>"," ",text)
    text = text.strip().lower()
    return text

In [7]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [8]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [9]:
#Tokenize

tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [10]:
#raw Data => tokenized inputs for finr-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [11]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [13]:
#working with our model
#NLP=>generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

C:\Users\kkris\AppData\Roaming\Python\Python310\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kkris\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1720.61it/s]


In [14]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("device:",device)

model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [15]:
#Training Arguments:
training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
)

trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [ ]:
#train the model

trainer.train()

C:\ProgramData\anaconda3\envs\ai_core\lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
#save the model
model.save_pretrained("./saved_summary_model")
tokenize.save_pretrained("./saved_summary_model")

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

In [ ]:
#Test the core logic for Summarization

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)
    #tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt").to(device)

    #generate the summary => token_id
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True)

    #decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens = True)
    return summary


    
